# Paired Comparison of the Seven Frozen BRFSS Models

This notebook performs **evaluation-only regeneration** of the seven already-frozen primary models.

No feature, preprocessing rule, hyperparameter, class weight, resampling method, or decision threshold is changed.

## Goals

1. regenerate row-level test predictions from every frozen model;
2. verify regenerated point metrics against the previously frozen results;
3. perform paired bootstrap comparison of ROC-AUC, PR-AUC, and F1;
4. perform pairwise McNemar tests on classification errors;
5. export comparison tables and overlaid ROC/PR curves.

Because every model is evaluated on the **same 68,508 respondents**, paired comparison is more informative than comparing independent confidence intervals.


In [ ]:
from pathlib import Path
from itertools import combinations
import glob
import json
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb

from scipy.stats import binomtest
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, precision_recall_curve
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

OUT = Path("/kaggle/working/primary_benchmark_comparison")
OUT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "DIABETE4"

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]
CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]
NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

xpt_candidates = glob.glob("/kaggle/input/**/LLCP2025.XPT", recursive=True)
if len(xpt_candidates) != 1:
    raise RuntimeError(f"Expected one XPT input, found {xpt_candidates}")
xpt_path = Path(xpt_candidates[0])

print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)


In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)
    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])
    out["_BMI5"] = raw["_BMI5"] / 100.0

    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)

    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)
    return out[PRIMARY_FEATURES]

target_mask = df[TARGET].isin([1.0, 3.0])
X = recode_brfss_2025(df.loc[target_mask, PRIMARY_FEATURES].copy())
y = (df.loc[target_mask, TARGET] == 1.0).astype("int8").rename("diabetes")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

assert len(X_train) == 274_031 and len(X_test) == 68_508
print("Train:", len(X_train), "Test:", len(X_test))


In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
])

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)

dense_transformer = FunctionTransformer(to_dense, accept_sparse=True)

models = {
    "logistic_regression": Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", LogisticRegression(
            penalty="l2", C=1.0, solver="lbfgs",
            max_iter=1000, class_weight=None
        )),
    ]),
    "decision_tree": Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", DecisionTreeClassifier(
            criterion="gini", splitter="best", max_depth=None,
            min_samples_split=2, min_samples_leaf=1, max_features=None,
            class_weight=None, ccp_alpha=0.0, random_state=42
        )),
    ]),
    "random_forest": Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", RandomForestClassifier(
            n_estimators=100, criterion="gini", max_depth=None,
            min_samples_split=2, min_samples_leaf=1, max_features="sqrt",
            bootstrap=True, class_weight=None, ccp_alpha=0.0,
            random_state=42, n_jobs=-1
        )),
    ]),
    "gradient_boosting": Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", GradientBoostingClassifier(
            loss="log_loss", learning_rate=0.1, n_estimators=100,
            subsample=1.0, criterion="friedman_mse",
            min_samples_split=2, min_samples_leaf=1, max_depth=3,
            max_features=None, random_state=42
        )),
    ]),
    "xgboost": Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", XGBClassifier(
            objective="binary:logistic", eval_metric="logloss",
            booster="gbtree", n_estimators=100, learning_rate=0.3,
            max_depth=6, min_child_weight=1, gamma=0.0,
            subsample=1.0, colsample_bytree=1.0,
            reg_alpha=0.0, reg_lambda=1.0, tree_method="hist",
            random_state=42, n_jobs=-1, verbosity=0
        )),
    ]),
    "linear_svm": Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", LinearSVC(
            penalty="l2", loss="squared_hinge", dual="auto",
            tol=1e-4, C=1.0, fit_intercept=True, class_weight=None,
            random_state=42, max_iter=10000
        )),
    ]),
    "gaussian_naive_bayes": Pipeline([
        ("preprocess", clone(preprocessor)),
        ("to_dense", dense_transformer),
        ("model", GaussianNB(priors=None, var_smoothing=1e-9)),
    ]),
}
print("Frozen model count:", len(models))


## 1. Regenerate and Verify Frozen Predictions

Every model is fitted once on the frozen full training set.

The regenerated metrics must match the already-frozen test results within numerical tolerance. A mismatch stops the notebook before any statistical comparison is performed.


In [ ]:
frozen = {
    "logistic_regression": {"accuracy":0.8558270566941087,"f1":0.2831845562087234,"roc_auc":0.8261678122652486,"pr_auc":0.44663290841217973},
    "decision_tree": {"accuracy":0.78615636130087,"f1":0.32351311414850387,"roc_auc":0.6020699144719784,"pr_auc":0.20533779176339034},
    "random_forest": {"accuracy":0.8510392946809132,"f1":0.2773174704341052,"roc_auc":0.8044953607113304,"pr_auc":0.40246501462675927},
    "gradient_boosting": {"accuracy":0.8563525427687277,"f1":0.26630880489077763,"roc_auc":0.8260155801805384,"pr_auc":0.4483169949856431},
    "xgboost": {"accuracy":0.8548198750510889,"f1":0.2904836638607505,"roc_auc":0.8262898793557036,"pr_auc":0.44171377834099224},
    "linear_svm": {"accuracy":0.8559876218835757,"f1":0.20242522231204527,"roc_auc":0.8261580968770885,"pr_auc":0.44788405460506064},
    "gaussian_naive_bayes": {"accuracy":0.7272873241081333,"f1":0.4400083925306477,"roc_auc":0.7885646728624294,"pr_auc":0.36412201184388493},
}

predictions = pd.DataFrame({
    "test_row_id": X_test.index.to_numpy(),
    "y_true": y_test.to_numpy(),
})
verification_rows = []

for name, pipe in models.items():
    print("Fitting", name)
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test).astype(int)
    if name == "linear_svm":
        score = pipe.decision_function(X_test)
        predictions[f"{name}_score"] = score
    else:
        score = pipe.predict_proba(X_test)[:, 1]
        predictions[f"{name}_prob"] = score

    predictions[f"{name}_pred"] = pred

    current = {
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, score),
        "pr_auc": average_precision_score(y_test, score),
    }
    row = {"model": name, **current}
    for metric, expected in frozen[name].items():
        delta = abs(current[metric] - expected)
        row[f"{metric}_abs_delta"] = delta
        if delta > 5e-7:
            raise RuntimeError(
                f"{name} regenerated {metric}={current[metric]} "
                f"does not match frozen {expected}; abs delta={delta}"
            )
    verification_rows.append(row)

verification = pd.DataFrame(verification_rows)
display(verification)
verification.to_csv(OUT / "regeneration_verification.csv", index=False)
predictions.to_csv(OUT / "paired_test_predictions.csv", index=False)

print("All seven frozen models reproduced within tolerance.")


## 2. Paired Bootstrap Differences

We resample the **same test respondents** for all models in each bootstrap replicate.

This preserves pairing and directly estimates the distribution of model-to-model metric differences.

Metrics:
- ROC-AUC;
- PR-AUC;
- F1 at each model's frozen/native decision boundary.

For each pair, we report:

```text
difference = metric(model_A) - metric(model_B)
```

A 95% bootstrap interval containing zero indicates that the observed difference is not clearly separated from zero under this resampling analysis.


In [ ]:
model_names = list(models)
y_np = predictions["y_true"].to_numpy()

scores = {}
preds = {}
for name in model_names:
    score_col = f"{name}_score" if name == "linear_svm" else f"{name}_prob"
    scores[name] = predictions[score_col].to_numpy()
    preds[name] = predictions[f"{name}_pred"].to_numpy()

pos_idx = np.flatnonzero(y_np == 1)
neg_idx = np.flatnonzero(y_np == 0)

BOOT_REPS = 200
BOOT_SEED = 20251010
rng = np.random.default_rng(BOOT_SEED)

records = []
for b in range(BOOT_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(idx)
    y_b = y_np[idx]

    for name in model_names:
        records.append({
            "replicate": b + 1,
            "model": name,
            "roc_auc": roc_auc_score(y_b, scores[name][idx]),
            "pr_auc": average_precision_score(y_b, scores[name][idx]),
            "f1": f1_score(y_b, preds[name][idx]),
        })

boot = pd.DataFrame(records)
boot.to_csv(OUT / "paired_bootstrap_model_metrics.csv", index=False)

pair_rows = []
for a, b in combinations(model_names, 2):
    a_df = boot[boot.model == a].sort_values("replicate")
    b_df = boot[boot.model == b].sort_values("replicate")
    for metric in ["roc_auc", "pr_auc", "f1"]:
        diff = a_df[metric].to_numpy() - b_df[metric].to_numpy()
        point_a = (
            roc_auc_score(y_np, scores[a]) if metric=="roc_auc"
            else average_precision_score(y_np, scores[a]) if metric=="pr_auc"
            else f1_score(y_np, preds[a])
        )
        point_b = (
            roc_auc_score(y_np, scores[b]) if metric=="roc_auc"
            else average_precision_score(y_np, scores[b]) if metric=="pr_auc"
            else f1_score(y_np, preds[b])
        )
        pair_rows.append({
            "model_a": a,
            "model_b": b,
            "metric": metric,
            "point_difference_a_minus_b": point_a - point_b,
            "bootstrap_mean_difference": float(diff.mean()),
            "ci95_lower": float(np.quantile(diff, 0.025)),
            "ci95_upper": float(np.quantile(diff, 0.975)),
            "ci_excludes_zero": bool((np.quantile(diff,0.025)>0) or (np.quantile(diff,0.975)<0)),
        })

paired_bootstrap = pd.DataFrame(pair_rows)
paired_bootstrap.to_csv(OUT / "paired_bootstrap_differences.csv", index=False)

display(
    paired_bootstrap[
        paired_bootstrap["model_a"].isin(["logistic_regression","gradient_boosting","xgboost","linear_svm"]) &
        paired_bootstrap["model_b"].isin(["logistic_regression","gradient_boosting","xgboost","linear_svm"]) &
        (paired_bootstrap.metric == "roc_auc")
    ]
)


## 3. McNemar Tests for Paired Classification Errors

McNemar tests compare whether two classifiers have the same error rate on the same respondents.

For models A and B:

- `b` = A correct, B wrong;
- `c` = A wrong, B correct.

A two-sided exact binomial test is applied to the discordant predictions.

Because 21 model pairs are tested, p-values are adjusted using the **Holm family-wise error correction**.


In [ ]:
mcnemar_rows = []
correct = {name: (preds[name] == y_np) for name in model_names}

for a, b in combinations(model_names, 2):
    b_count = int(np.sum(correct[a] & ~correct[b]))
    c_count = int(np.sum(~correct[a] & correct[b]))
    discordant = b_count + c_count

    p = 1.0 if discordant == 0 else binomtest(
        k=min(b_count, c_count),
        n=discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue

    mcnemar_rows.append({
        "model_a": a,
        "model_b": b,
        "a_correct_b_wrong": b_count,
        "a_wrong_b_correct": c_count,
        "discordant_n": discordant,
        "p_value_exact": p,
    })

mcnemar = pd.DataFrame(mcnemar_rows).sort_values("p_value_exact").reset_index(drop=True)

# Holm adjustment
m = len(mcnemar)
raw = mcnemar["p_value_exact"].to_numpy()
adjusted_sorted = np.maximum.accumulate(
    np.minimum(1.0, (m - np.arange(m)) * raw)
)
mcnemar["p_value_holm"] = adjusted_sorted
mcnemar["significant_holm_0_05"] = mcnemar["p_value_holm"] < 0.05

mcnemar.to_csv(OUT / "mcnemar_pairwise.csv", index=False)
display(mcnemar)


## 4. Overlay Curves

ROC and precision-recall curves are plotted from the same paired test predictions.

These plots provide threshold-independent visual comparison; they do not change any model or threshold.


In [ ]:
display_names = {
    "logistic_regression":"Logistic Regression",
    "decision_tree":"Decision Tree",
    "random_forest":"Random Forest",
    "gradient_boosting":"Gradient Boosting",
    "xgboost":"XGBoost",
    "linear_svm":"Linear SVM",
    "gaussian_naive_bayes":"Gaussian NB",
}

plt.figure(figsize=(8,6))
for name in model_names:
    fpr, tpr, _ = roc_curve(y_np, scores[name])
    auc = roc_auc_score(y_np, scores[name])
    plt.plot(fpr, tpr, label=f"{display_names[name]} ({auc:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Seven Frozen Models — ROC Curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "roc_overlay.png", dpi=180)
plt.show()

plt.figure(figsize=(8,6))
for name in model_names:
    p, r, _ = precision_recall_curve(y_np, scores[name])
    ap = average_precision_score(y_np, scores[name])
    plt.plot(r, p, label=f"{display_names[name]} ({ap:.3f})")
plt.axhline(y=y_np.mean(), linestyle="--", label=f"Prevalence ({y_np.mean():.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Seven Frozen Models — Precision-Recall Curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "pr_overlay.png", dpi=180)
plt.show()


## 5. Export Evaluation Manifest

This manifest records that the comparison is evaluation-only and uses the previously frozen specifications.


In [ ]:
manifest = {
    "purpose": "paired evaluation of seven frozen primary models",
    "test_n": int(len(y_test)),
    "bootstrap_reps": BOOT_REPS,
    "bootstrap_seed": BOOT_SEED,
    "mcnemar_correction": "Holm FWER",
    "models": model_names,
    "model_retuning": False,
    "threshold_optimization": False,
    "test_set_used_for_training": False,
    "python": platform.python_version(),
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
}
with open(OUT / "comparison_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))
